In [1]:
!pip install -qU langchain langgraph langchain-community langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
import langchain
import langgraph
import langchain_community
import langchain_google_genai
print("LangChain: " ,langchain.__version__)
print("All required libraries imported successfully!")

/tmp/ipykernel_1748/3478340345.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


LangChain:  1.3.14
All required libraries imported successfully!


In [8]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

response = llm.invoke("What is SQL?")
print(response.content)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': '**SQL** (pronounced *“S-Q-L”* or *“Sequel”*) stands for **Structured Query Language**. \n\nIt is the standard programming language used to communicate with, manage, and manipulate **relational databases**. \n\nIf you think of a database as a massive, digital filing cabinet (or a collection of connected Excel spreadsheets), SQL is the language you use to store information, search through it, update it, or delete it.\n\n---\n\n### What Does SQL Do?\nSQL allows you to perform basic operations often referred to as **CRUD**:\n\n1. **C**reate: Add new data to a database.\n2. **R**ead: Search and retrieve specific data.\n3. **U**pdate: Edit or modify existing data.\n4. **D**elete: Remove data you no longer need.\n\n### A Simple Example\nSQL is designed to read somewhat like natural English. \n\nImagine you have a database table named `Customers`. If you want to find the names and email addresses of all customers who live in "New York", the SQL code looks like this:\

In [13]:
import sqlite3

conn = sqlite3.connect("shop.db")
cursor = conn.cursor()
print("data base sucessfully connected")

data base sucessfully connected


In [14]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    price REAL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    amount REAL,
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
)
""")

conn.commit()

In [15]:
customers = [
    (1, "Manoj", "Hyderabad"),
    (2, "Rahul", "Bangalore"),
    (3, "Suresh", "Chennai"),
    (4, "Priya", "Hyderabad")
]

products = [
    (1, "Laptop", 55000),
    (2, "Smartphone", 25000),
    (3, "Headphones", 3000),
    (4, "Keyboard", 2000)
]

orders = [
    (1, 1, 1, 1, 55000),
    (2, 1, 3, 2, 6000),
    (3, 2, 2, 1, 25000),
    (4, 3, 1, 2, 110000),
    (5, 3, 4, 2, 4000),
    (6, 4, 2, 1, 25000)
]

cursor.executemany(
    "INSERT OR IGNORE INTO customers VALUES (?, ?, ?)",
    customers
)

cursor.executemany(
    "INSERT OR IGNORE INTO products VALUES (?, ?, ?)",
    products
)

cursor.executemany(
    "INSERT OR IGNORE INTO orders VALUES (?, ?, ?, ?, ?)",
    orders
)

conn.commit()

print("Database created successfully!")

Database created successfully!


In [16]:
cursor.execute("SELECT * FROM customers")
print(cursor.fetchall())

[(1, 'Manoj', 'Hyderabad'), (2, 'Rahul', 'Bangalore'), (3, 'Suresh', 'Chennai'), (4, 'Priya', 'Hyderabad')]


In [17]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///shop.db")

print("Database dialect:", db.dialect)
print("Tables:", db.get_usable_table_names())

Database dialect: sqlite
Tables: ['customers', 'orders', 'products']


In [18]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)

tools = toolkit.get_tools()

for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [19]:
from langchain.agents import create_agent

system_prompt = """
You are a SQL database assistant.

Your job is to answer questions about the shop database.

Rules:
1. First inspect the available tables.
2. Then inspect the relevant table schemas.
3. Generate a correct SQLite SQL query.
4. Check the SQL query before executing it.
5. Only perform SELECT queries.
6. Never INSERT, UPDATE, DELETE or DROP data.
7. Return a simple natural-language answer.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("SQL Agent created successfully!")

SQL Agent created successfully!


In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

response = llm.invoke("Say hello in one sentence.")

print(response.content)

[{'type': 'text', 'text': 'Hello, how can I help you today?', 'extras': {'signature': 'EjQKMgERTTIPqrB0a9/7zyK/n620AegNdL7zNQxPVzw2G35pj4lPY0aglXWBmnJ9cnFlqVyC'}}]


In [22]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)

tools = toolkit.get_tools()

print("SQL tools recreated successfully!")

for tool in tools:
    print("-", tool.name)

SQL tools recreated successfully!
- sql_db_query
- sql_db_schema
- sql_db_list_tables
- sql_db_query_checker


In [23]:
from langchain.agents import create_agent

system_prompt = """
You are a helpful SQL database assistant.

You have access to a shop database.

Follow these rules:
1. First identify the relevant tables.
2. Check the table schema before writing SQL.
3. Generate a valid SQLite SQL query.
4. Check the SQL query before executing it.
5. Only use SELECT queries.
6. Never modify or delete database data.
7. After getting the result, explain the answer clearly.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

In [24]:
question = "Who spent the most money?"

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': ' Suresh spent the most money, with a total of 114,000.0.', 'extras': {'signature': 'EjQKMgERTTIPYzHtDnOmnE8KyuZ5nqrUvm9kaA9dyBU9ey7qdGdJswRlywkHIIJhwo8uvPfp'}}]


In [25]:
for i, message in enumerate(result["messages"]):
    print(f"\n--- Step {i + 1} ---")
    print("Type:", type(message).__name__)
    print(message)


--- Step 1 ---
Type: HumanMessage
content='Who spent the most money?' additional_kwargs={} response_metadata={} id='e76be105-a29e-4db4-a778-00548b21a9f1'

--- Step 2 ---
Type: AIMessage
content=[] additional_kwargs={'function_call': {'name': 'sql_db_list_tables', 'arguments': '{"tool_input": ""}'}, '__gemini_function_call_thought_signatures__': {'sJmNDmkf': 'EjQKMgERTTIPom+vuyjhVLSgit17RneqEOss+lTtzBfHmdX412xVzpsShC+C679r9gtvg0Ur'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019ff01f-96a3-7952-8004-59402c506849-0' tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'sJmNDmkf', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 514, 'output_tokens': 21, 'total_tokens': 535, 'input_token_details': {'cache_read': 0}}

--- Step 3 ---
Type: ToolMessage
content='customers, orders, products' name='sql_db_list_tables' id='8ddcfccb-aff1-4